# Final Evaluation of baseline models on the test set
The role of this notebook is to load the best model for tabular baselines (Cox PH and RSF), load the test set and evaluate the performance.

In [ ]:
# Project content directory 
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import importlib
import src.config
import src.data_loading
importlib.reload(src.config)
importlib.reload(src.data_loading)

In [ ]:
from src.config import (
    X_TEST_TRUNC_WS_FILE,
    X_TEST_TRUNC_NS_FILE,
    Y_TEST_TRUNC_FILE,
    Y_TEST_TRUNC_FILE,
    Y_TRAIN_TRUNC_FILE,
    Y_TRAIN_FILE
)

In [ ]:
from src.data_loading import load_csv
y_train_trunc = load_csv(Y_TRAIN_TRUNC_FILE, "y_train_trunc")

In [ ]:
y_train = load_csv(Y_TRAIN_FILE, "y_train")

In [ ]:
y_train.head()

In [ ]:
y_train_trunc.head()

In [ ]:
from src.data_loading import load_processed_training_data

X_test_trunc_with_static, X_test_trunc_no_static, y_test_trunc, y_test_trunc = load_processed_training_data(
    X_TEST_TRUNC_WS_FILE,
    X_TEST_TRUNC_NS_FILE,
    Y_TEST_TRUNC_FILE,
    Y_TEST_TRUNC_FILE
)

In [ ]:
import pickle
from pathlib import Path
import pandas as pd
from sksurv.metrics import concordance_index_censored

RESULTS_DIR = PROJECT_ROOT / "results" / "baselines_trunc"

def load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)

In [ ]:
# load saved results
cph_ws = load_pickle(RESULTS_DIR / "cph_trunc_with_static.pkl")
cph_ns = load_pickle(RESULTS_DIR / "cph_trunc_without_static.pkl")

rsf_ws = load_pickle(RESULTS_DIR / "rsf_trunc_with_static.pkl")
rsf_ns = load_pickle(RESULTS_DIR / "rsf_trunc_no_static.pkl")

In [ ]:
# inspect keys
print(cph_ws.keys())
print(rsf_ws.keys())
print(rsf_ws["tuned"].keys())

In [ ]:
# rsf evaluation with static
rsf_model_ws = rsf_ws["tuned"]["fitted_model"]

rsf_risk_ws = rsf_model_ws.predict(X_test_trunc_with_static)

rsf_test_cindex_ws = concordance_index_censored(
    y_test_trunc["event"].astype(bool),
    y_test_trunc["duration"],
    rsf_risk_ws,
)[0]

print("RSF with static test C-index:", rsf_test_cindex_ws)

In [ ]:
# rsf evaluation without static
rsf_model_ns = rsf_ns["tuned"]["fitted_model"]

rsf_risk_ns = rsf_model_ns.predict(X_test_trunc_no_static)

rsf_test_cindex_ns = concordance_index_censored(
    y_test_trunc["event"].astype(bool),
    y_test_trunc["duration"],
    rsf_risk_ns,
)[0]

print("RSF without static test C-index:", rsf_test_cindex_ns)

In [ ]:
# cox, with static
cph_model_ws = cph_ws["fitted_model"]

# Using log-risk / linear predictor instead of exp(linear predictor)
cph_risk_ws = cph_model_ws.predict_log_partial_hazard(
    X_test_trunc_with_static
).values.ravel()

cph_test_cindex_ws = concordance_index_censored(
    y_test_trunc["event"].astype(bool).values,
    y_test_trunc["duration"].values,
    cph_risk_ws,
)[0]

print("Cox PH with static test C-index:", cph_test_cindex_ws)

In [ ]:
# cox, without static
cph_model_ns = cph_ns["fitted_model"]

# Using log-risk / linear predictor instead of exp(linear predictor)
cph_risk_ns = cph_model_ns.predict_log_partial_hazard(
    X_test_trunc_no_static
).values.ravel()

cph_test_cindex_ns = concordance_index_censored(
    y_test_trunc["event"].astype(bool).values,
    y_test_trunc["duration"].values,
    cph_risk_ns,
)[0]

print("Cox PH without static test C-index:", cph_test_cindex_ns)

In [ ]:
# table of final results
final_test_results = pd.DataFrame(
    [
        {
            "model": "Cox PH without static",
            "test_c_index": cph_test_cindex_ns,
        },
        {
            "model": "Cox PH with static",
            "test_c_index": cph_test_cindex_ws,
        },
        {
            "model": "RSF without static",
            "test_c_index": rsf_test_cindex_ns,
        },
        {
            "model": "RSF with static",
            "test_c_index": rsf_test_cindex_ws,
        },
        {
            "model": "SSL TFC static MLP16",
            "test_c_index": 0.727424,
        },
    ]
)

final_test_results = final_test_results.sort_values(
    "test_c_index",
    ascending=False,
)

print(final_test_results)

In [ ]:
# save final results
RESULTS_DIR = PROJECT_ROOT / "results" / "final"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

final_results_df = pd.DataFrame(
    [
        {"model": "SSL TFC static MLP16", "test_c_index": 0.727424},
        {"model": "RSF with static", "test_c_index": 0.716737},
        {"model": "RSF without static", "test_c_index": 0.710215},
        {"model": "Cox PH without static", "test_c_index": 0.664538},
        {"model": "Cox PH with static", "test_c_index": 0.658040},
    ]
)

final_results_df = final_results_df.sort_values(
    "test_c_index",
    ascending=False
)

csv_path = RESULTS_DIR / "final_test_results.csv"
final_results_df.to_csv(csv_path, index=False)

print("Saved CSV:", csv_path)
print(final_results_df)

In [ ]:
# save JSON for reproducibility
import json

json_path = RESULTS_DIR / "final_test_results.json"

with open(json_path, "w") as f:
    json.dump(
        final_results_df.to_dict(orient="records"),
        f,
        indent=4
    )

print("Saved JSON:", json_path)

## Computing additional evaluation metrics: time-dependent AUC and IBS
For a more complete comparison, more metrics are needed. The IBS that examines how accurate were predicted survival probabilities and a time dependent AUC that shows how well the model could predict risk at specific time horizons.

In [ ]:
import numpy as np

times = np.linspace(
    np.percentile(y_test_trunc["duration"], 5),
    np.percentile(y_test_trunc["duration"], 95),
    100,
)

times = times[
    (times > y_train_trunc["duration"].min()) &
    (times < y_train_trunc["duration"].max())
]

In [ ]:
# retrieve survival functions from rsf 
rsf_model = rsf_ws["tuned"]["fitted_model"]

surv_funcs = rsf_model.predict_survival_function(X_test_trunc_with_static)

In [ ]:
# convert to survival matrix
surv_matrix_rsf = np.vstack([
    fn(times) for fn in surv_funcs
])

In [ ]:
# structured arrays
from sksurv.util import Surv

y_train_struct = Surv.from_arrays(
    event=y_train_trunc["event"].astype(bool),
    time=y_train_trunc["duration"]
)

y_test_struct = Surv.from_arrays(
    event=y_test_trunc["event"].astype(bool),
    time=y_test_trunc["duration"]
)

In [ ]:
# compute AUC metric from RSF
from sksurv.metrics import cumulative_dynamic_auc

rsf_risk = rsf_model.predict(X_test_trunc_with_static)

auc_rsf, mean_auc_rsf = cumulative_dynamic_auc(
    y_train_struct,
    y_test_struct,
    rsf_risk,
    times
)

In [ ]:
# compute IBS metric from RSF
from sksurv.metrics import integrated_brier_score

ibs_rsf = integrated_brier_score(
    y_train_struct,
    y_test_struct,
    surv_matrix_rsf,
    times
)


In [ ]:
# define model
cph_model = cph_ws["fitted_model"]

# log-risk / linear predictor
cph_risk = cph_model.predict_log_partial_hazard(
    X_test_trunc_with_static
).values.ravel()

In [ ]:
# compute AUC from cph
auc_cph, mean_auc_cph = cumulative_dynamic_auc(
    y_train_struct,
    y_test_struct,
    cph_risk,
    times
)

In [ ]:
baseline_times = cph_model.baseline_cumulative_hazard_.index.values.astype(float)

test_min = y_test_trunc["duration"].min()
test_max = y_test_trunc["duration"].max()

train_min = y_train_trunc["duration"].min()
train_max = y_train_trunc["duration"].max()

cox_times = baseline_times[
    (baseline_times >= test_min) &
    (baseline_times < test_max) &
    (baseline_times > train_min) &
    (baseline_times < train_max)
]

# Optional: reduce to 100 points if too many
if len(cox_times) > 100:
    idx = np.linspace(0, len(cox_times) - 1, 100).astype(int)
    cox_times = cox_times[idx]

In [ ]:
surv_df = cph_model.predict_survival_function(
    X_test_trunc_with_static,
    times=cox_times,
)

surv_matrix_cph = surv_df.values.T

ibs_cph = integrated_brier_score(
    survival_train=y_train_struct,
    survival_test=y_test_struct,
    estimate=surv_matrix_cph,
    times=cox_times,
)

print("Cox IBS:", ibs_cph)

In [ ]:
cph_model_ns = cph_ns["fitted_model"]

# Risk scores for C-index / AUC
cph_risk_ns = cph_model_ns.predict_log_partial_hazard(
    X_test_trunc_no_static
).values.ravel()

# Survival functions for IBS
baseline_times = cph_model_ns.baseline_cumulative_hazard_.index.values.astype(float)

test_min = y_test_trunc["duration"].min()
test_max = y_test_trunc["duration"].max()
train_min = y_train_trunc["duration"].min()
train_max = y_train_trunc["duration"].max()

cox_times = baseline_times[
    (baseline_times >= test_min) &
    (baseline_times < test_max) &
    (baseline_times > train_min) &
    (baseline_times < train_max)
]

if len(cox_times) > 100:
    idx = np.linspace(0, len(cox_times) - 1, 100).astype(int)
    cox_times = cox_times[idx]

surv_df_ns = cph_model_ns.predict_survival_function(
    X_test_trunc_no_static,
    times=cox_times,
)

surv_matrix_cph_ns = surv_df_ns.values.T

print("Survival min/max:", surv_matrix_cph_ns.min(), surv_matrix_cph_ns.max())
print("Mean survival first:", surv_matrix_cph_ns[:, 0].mean())
print("Mean survival last:", surv_matrix_cph_ns[:, -1].mean())

ibs_cph_ns = integrated_brier_score(
    survival_train=y_train_struct,
    survival_test=y_test_struct,
    estimate=surv_matrix_cph_ns,
    times=cox_times,
)

print("Cox PH without static IBS:", ibs_cph_ns)

In [ ]:
import pandas as pd

final_metrics_tab_baselines = pd.DataFrame([
    {
        "model": "RSF with static",
        "c_index": rsf_test_cindex_ws,
        "mean_auc": mean_auc_rsf,
        "ibs": ibs_rsf,
    },
    {
        "model": "Cox PH",
        "c_index": cph_test_cindex_ws,
        "mean_auc": mean_auc_cph,
        "ibs": ibs_cph,
    },
])

final_metrics = final_metrics_tab_baselines.sort_values("c_index", ascending=False)
print(final_metrics)